# LMA Phase 2 - Kaggle training and evaluation

Attach the private dataset created from `lma_phase2_kaggle_upload.zip`, select a GPU accelerator, and run the cells in order. Checkpoints are written to `/kaggle/working/phase2_runs`.

In [ ]:
import subprocess
subprocess.run(['nvidia-smi'], check=True)

In [ ]:
from pathlib import Path
import shutil

manifests = sorted(Path('/kaggle/input').rglob('lma_phase2_bundle_manifest.json'))
if manifests:
    BUNDLE_ROOT = manifests[0].parent
else:
    archives = sorted(Path('/kaggle/input').rglob('lma_phase2_kaggle_upload.zip'))
    if not archives:
        raise FileNotFoundError('Attach the uploaded LMA Phase 2 Kaggle dataset first')
    extraction_root = Path('/kaggle/working/lma_phase2_bundle')
    extraction_root.mkdir(parents=True, exist_ok=True)
    shutil.unpack_archive(str(archives[0]), extraction_root)
    manifests = sorted(extraction_root.rglob('lma_phase2_bundle_manifest.json'))
    BUNDLE_ROOT = manifests[0].parent
RUNNER = BUNDLE_ROOT / 'kaggle' / 'run_phase2_kaggle.py'
print('Bundle:', BUNDLE_ROOT)
print('Runner:', RUNNER)

## Train or resume

Use `both` to train Hindi followed by Bhojpuri. To reduce the risk of hitting the session limit, you may run one language per session. If a previous notebook output is attached, set `RESUME_ROOT` to the directory containing its `phase2_runs` folder.

In [ ]:
LANGUAGE = 'both'  # 'hindi', 'bhojpuri', or 'both'
RESUME_ROOT = None  # e.g. '/kaggle/input/previous-phase2-output'
command = [
    'python', str(RUNNER),
    '--bundle-root', str(BUNDLE_ROOT),
    '--language', LANGUAGE,
    '--mode', 'train',
    '--device', 'cuda',
]
if RESUME_ROOT:
    command.extend(['--resume-root', RESUME_ROOT])
subprocess.run(command, check=True)

## Evaluate completed checkpoints

This computes held-out CE/PPL/BPB, generation metrics and samples, plus attention heatmaps and summaries. Kaggle internet must be enabled if the two small evaluation packages are missing.

In [ ]:
import importlib.util
missing = [name for name in ('sentencepiece', 'sacrebleu') if importlib.util.find_spec(name) is None]
if missing:
    subprocess.run(['python', '-m', 'pip', 'install', '-q', *missing], check=True)
subprocess.run([
    'python', str(RUNNER),
    '--bundle-root', str(BUNDLE_ROOT),
    '--language', 'both',
    '--mode', 'evaluate',
    '--device', 'cuda',
], check=True)

## Package outputs

Download this archive and/or save the notebook as a version so the checkpoints can be attached to a later session.

In [ ]:
import zipfile
output_zip = Path('/kaggle/working/lma_phase2_outputs.zip')
with zipfile.ZipFile(output_zip, 'w', compression=zipfile.ZIP_STORED) as archive:
    report_dir = Path('/kaggle/working/phase2_report')
    if report_dir.exists():
        for path in report_dir.rglob('*'):
            if path.is_file():
                archive.write(path, path.relative_to('/kaggle/working'))
    runs_dir = Path('/kaggle/working/phase2_runs')
    for run_dir in runs_dir.glob('*') if runs_dir.exists() else []:
        metadata_names = ('latest.json', 'training_log.csv', 'resolved_config.json', 'run_environment.json')
        for name in metadata_names:
            path = run_dir / name
            if path.exists():
                archive.write(path, path.relative_to('/kaggle/working'))
        latest_path = run_dir / 'latest.json'
        if latest_path.exists():
            latest = __import__('json').loads(latest_path.read_text())
            checkpoint = run_dir / latest['checkpoint']
            archive.write(checkpoint, checkpoint.relative_to('/kaggle/working'))
print(output_zip, output_zip.stat().st_size, 'bytes')